# **6. Временная кросс-валидация и подбор гиперпараметров (Time Series Cross-Validation and Hyperparameter Tuning)**

* __Цель:__ настроить основные модели в expanding-window схеме без утечки информации из будущих периодов.
* __Задачи:__
  - построить `TimeSeriesSplit` для train-выборки;
  - объединить preprocessing и estimator в sklearn Pipeline;
  - выполнить компактный `RandomizedSearchCV` по RMSE;
  - сохранить CV-результаты и лучшие параметры;
  - сравнить default и tuned конфигурации на внешней validation-выборке;
  - сохранить global test split закрытым.
* __Алгоритм выполнения:__
  1. Подготовить feature dataset и хронологические split.
  2. Проверить expanding-window folds.
  3. Изучить модели и compact search spaces.
  4. Запустить fold-safe tuning pipeline.
  5. Сопоставить default и tuned validation metrics.
  6. Выполнить methodological audit.

In [ ]:
import json

import pandas as pd
from IPython.display import Markdown, display

from traffic_forecasting.config import DATETIME_COLUMN, TIME_SERIES_CV_SPLITS, TUNING_N_ITER
from traffic_forecasting.data_loader import load_raw_data
from traffic_forecasting.evaluation import (
    build_time_series_split,
    summarize_time_series_splits,
)
from traffic_forecasting.features import build_feature_dataset
from traffic_forecasting.models import (
    get_hyperparameter_search_spaces,
    get_tuning_model_registry,
)
from traffic_forecasting.pipeline import run_model_tuning_pipeline
from traffic_forecasting.preprocessing import prepare_model_inputs, split_chronologically

## **6.1. Подготовка хронологических данных (Chronological Data Preparation)**

In [ ]:
feature_data = build_feature_dataset(load_raw_data())
train_data, validation_data, test_data = split_chronologically(feature_data)
(
    X_train,
    X_validation,
    _X_test,
    y_train,
    y_validation,
    _y_test,
    timestamps_train,
    timestamps_validation,
    timestamps_test,
) = prepare_model_inputs(train_data, validation_data, test_data)

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(data),
            "start": data[DATETIME_COLUMN].min(),
            "end": data[DATETIME_COLUMN].max(),
        }
        for name, data in (
            ("train", train_data),
            ("validation", validation_data),
            ("test", test_data),
        )
    ]
)

display(Markdown("### **Временные границы global splits (Global Split Ranges)**"))
display(split_summary)

## **6.2. Схема expanding-window кросс-валидации (Expanding-Window Cross-Validation Scheme)**

In [ ]:
time_series_cv = build_time_series_split(TIME_SERIES_CV_SPLITS)
cv_summary = summarize_time_series_splits(len(X_train), time_series_cv)
cv_summary["train_end_timestamp"] = cv_summary["train_end"].map(
    lambda index: timestamps_train.iloc[index]
)
cv_summary["validation_start_timestamp"] = cv_summary["validation_start"].map(
    lambda index: timestamps_train.iloc[index]
)
cv_summary["validation_end_timestamp"] = cv_summary["validation_end"].map(
    lambda index: timestamps_train.iloc[index]
)

display(Markdown("### **Границы TimeSeriesSplit folds (TimeSeriesSplit Fold Boundaries)**"))
display(cv_summary)

## **6.3. Настраиваемые модели и search spaces (Tuned Models and Search Spaces)**

In [ ]:
tuning_registry = get_tuning_model_registry()
search_spaces = get_hyperparameter_search_spaces()
search_space_summary = pd.DataFrame(
    [
        {
            "model": model_name,
            "estimator": type(tuning_registry[model_name]).__name__,
            "search_parameters": ", ".join(parameter_space),
            "randomized_iterations": TUNING_N_ITER,
        }
        for model_name, parameter_space in search_spaces.items()
    ]
)

display(Markdown("### **Компактные пространства поиска (Compact Search Spaces)**"))
display(search_space_summary)

## **6.4. Time-series-aware подбор гиперпараметров (Time-Series-Aware Hyperparameter Tuning)**

In [ ]:
comparison, best_parameters, tuning_results = run_model_tuning_pipeline()

display(Markdown("### **Лучшие параметры по CV RMSE (Best Parameters by CV RMSE)**"))
display(best_parameters)
display(Markdown("### **Лучшие результаты search candidates (Best Search Candidates)**"))
display(tuning_results.query("rank == 1").sort_values("mean_cv_rmse"))

## **6.5. Сравнение default и tuned конфигураций (Default and Tuned Comparison)**

In [ ]:
default_metrics = comparison.query("configuration == 'default'").set_index("model")
tuned_metrics = comparison.query("configuration == 'tuned'").set_index("model")
rmse_comparison = pd.DataFrame(
    {
        "default_rmse": default_metrics["rmse"],
        "tuned_rmse": tuned_metrics["rmse"],
    }
)
rmse_comparison["rmse_change"] = rmse_comparison["tuned_rmse"] - rmse_comparison["default_rmse"]
rmse_comparison["tuning_improved_validation"] = rmse_comparison["rmse_change"] < 0
rmse_comparison = rmse_comparison.sort_values("tuned_rmse")

display(Markdown("### **Изменение validation RMSE после tuning (Validation RMSE Change)**"))
display(rmse_comparison)
display(Markdown("### **Рейтинг tuned-моделей (Tuned Model Ranking)**"))
display(tuned_metrics.sort_values("rmse").reset_index())

## **6.6. Аудит методологических ограничений (Methodological Constraints Audit)**

In [ ]:
decoded_parameters = best_parameters["best_parameters"].map(json.loads)
methodology_audit = pd.DataFrame(
    {
        "check": [
            "train_precedes_validation",
            "validation_precedes_locked_test",
            "cv_folds_are_expanding",
            "cv_train_precedes_fold_validation",
            "comparison_contains_no_test_metrics",
            "all_search_parameters_target_model_step",
            "randomized_search_is_compact",
        ],
        "passed": [
            timestamps_train.max() < timestamps_validation.min(),
            timestamps_validation.max() < timestamps_test.min(),
            cv_summary["train_size"].is_monotonic_increasing,
            (cv_summary["train_end"] < cv_summary["validation_start"]).all(),
            "split" not in comparison.columns,
            all(
                key.startswith("model__") for parameters in decoded_parameters for key in parameters
            ),
            TUNING_N_ITER <= 3,
        ],
    }
)

display(Markdown("### **Результаты methodological audit (Methodology Audit Results)**"))
display(methodology_audit)

assert bool(methodology_audit["passed"].all()), "Model tuning methodology audit failed."

## **6.7. Анализ и интерпретация результатов tuning (Analysis and Interpretation of Tuning Results)**

На этапе настройки гиперпараметров была реализована процедура time-series-aware tuning для выбранных регрессионных моделей прогнозирования транспортной нагрузки. Данный этап продолжает baseline- и ensemble-сравнение и направлен на уточнение качества моделей за счет подбора ключевых гиперпараметров с учетом временной природы данных.

**Ключевые результаты:**
1. **Реализована схема time-series-aware настройки гиперпараметров.**
   В рамках этапа была применена перекрестная проверка для временных рядов по схеме расширяющегося окна. На каждом fold модель обучалась на более раннем фрагменте временного ряда и проверялась на следующем по времени validation-блоке.
2. **Preprocessing встроен внутрь tuning pipeline.**
   Для настройки моделей использовался sklearn `Pipeline`, включающий два основных шага: предварительную обработку признаков и регрессионную модель. Это означает, что на каждом fold кросс-валидации preprocessing-конвейер обучался только на train-части текущего fold, после чего применялся к validation-части этого же fold. Такая реализация предотвращает validation-fold leakage, при котором статистики масштабирования, импутации или кодирования могли бы быть рассчитаны с использованием будущих данных.
3. **Настройка выполнялась по основной метрике `RMSE`.**
   В качестве основной objective metric для выбора лучших гиперпараметров использовалась минимизация `RMSE`. В реализации scikit-learn это соответствует метрике `neg_root_mean_squared_error`, так как процедуры `GridSearchCV` и `RandomizedSearchCV` максимизируют значение scoring-функции. Дополнительно для анализа результатов рассчитывались метрики `MAE`, `MAPE` и `R²`, что обеспечивает сопоставимость с baseline- и ensemble-этапами.
4. **Для настройки использованы компактные пространства гиперпараметров.**
   В tuning-этап были включены модели `Ridge`, `DecisionTreeRegressor`, `RandomForestRegressor`, `GradientBoostingRegressor`, `XGBRegressor`, `LGBMRegressor` и `CatBoostRegressor`. Для каждой модели были заданы компактные search spaces, включающие наиболее значимые параметры: регуляризацию для `Ridge`, глубину и минимальный размер листа для деревьев, число деревьев, learning rate, глубину, subsampling и параметры регуляризации для ансамблевых моделей.
5. **Сравнение default и tuned-конфигураций выполнено на validation-выборке.**
   После подбора гиперпараметров каждая tuned-модель была сопоставлена с соответствующей default-конфигурацией на validation-выборке. Такой подход позволяет оценить, улучшает ли настройка гиперпараметров качество модели относительно базовых параметров. В текущем compact tuning некоторые модели показали улучшение качества после настройки, тогда как для отдельных моделей tuned-конфигурация оказалась хуже default-варианта. Это является ожидаемым результатом при ограниченном числе случайных комбинаций и небольшом пространстве поиска.
6. **Наилучший tuned validation-result показала модель `XGBRegressor`.**
   По результатам validation-сравнения лучшую tuned-конфигурацию среди рассмотренных моделей показала модель `XGBRegressor`. Это указывает на то, что градиентный бустинг с подобранными параметрами способен эффективно учитывать нелинейные зависимости между временными, календарными, погодными, лаговыми и rolling-признаками. При этом результат tuning следует рассматривать как предварительный, поскольку число проверенных комбинаций гиперпараметров было ограничено.
7. **Улучшение после tuning наблюдалось не для всех моделей.**
   Для части моделей настройка позволила снизить ошибку прогноза на validation-выборке, однако для некоторых алгоритмов default-конфигурации сохранили более высокое качество. Это не означает, что данные модели менее применимы к задаче прогнозирования транспортной нагрузки. Скорее, это показывает, что компактный `RandomizedSearchCV` с малым числом итераций может не попасть в оптимальную область гиперпараметров.

**Итоговое методологическое резюме:** этап Time Series Cross-Validation and Hyperparameter Tuning реализовал корректную схему настройки гиперпараметров для моделей прогнозирования транспортной нагрузки с учетом временной структуры данных. Использование expanding-window `TimeSeriesSplit` и sklearn `Pipeline` позволило выполнить настройку без утечки информации между train- и validation-fold. Основной метрикой оптимизации выступала `RMSE`, а дополнительные метрики `MAE`, `MAPE` и `R²` использовались для комплексного анализа качества. Полученные результаты показывают, что настройка гиперпараметров может улучшать качество отдельных моделей, однако из-за компактного пространства поиска результаты следует рассматривать как предварительный tuning-этап.